# AI-Powered Event Analysis

This notebook uses the EQ Syntropy RAG (Retrieval-Augmented Generation) API
to analyze power quality events with AI assistance. It demonstrates:

1. Querying recent PQ events from the gateway API
2. Submitting events for AI analysis
3. Interpreting structured analysis results
4. Generating incident reports

**Prerequisites:**

- **From demo.pq.app JupyterLab**: Works automatically (RAG server is running).
- **From a local `equser` (PyPI) installation**: You need demo access credentials.
  Request access at [demo.pq.app](https://demo.pq.app). Then set `RAG_BASE_URL`
  below to point to the demo RAG endpoint.

## 1. Setup

In [ ]:
import json
import textwrap
from datetime import datetime, timedelta

import requests

from equser.api import SynapseClient

# Gateway API (data queries)
SYNAPSE_BASE_URL = 'http://127.0.0.1:8080'

# RAG API (AI analysis)
# When running inside demo.pq.app JupyterLab, this is localhost.
# When running from a remote equser installation, set this to
# the demo RAG endpoint provided with your access credentials.
RAG_BASE_URL = 'http://127.0.0.1:8081'

client = SynapseClient(SYNAPSE_BASE_URL)

# Verify connectivity
try:
    devices = client.list_devices()
    print(f'Connected to gateway API: {len(devices)} device(s)')
    for d in devices:
        print(f"  {d.get('id', 'unknown')}: {d.get('name', '')}")
except Exception as e:
    print(f'Gateway API not reachable: {e}')
    print('Ensure the Synapse backend is running.')

try:
    r = requests.get(f'{RAG_BASE_URL}/health', timeout=5)
    print(f'Connected to RAG API: {r.json()}')
except Exception as e:
    print(f'RAG API not reachable: {e}')
    print('AI analysis features will not be available.')
    print('If running locally, request demo access at https://demo.pq.app')

## 2. Query Recent Events

Fetch recent power quality events from the gateway. Events are anomalies
detected by the waveform processor (voltage sags, swells, harmonic distortion,
transients, frequency deviations).

In [ ]:
# Fetch recent events
try:
    events = client.get_events(limit=20)
    print(f'Found {len(events)} recent event(s)\n')

    for i, evt in enumerate(events[:10]):
        eid = evt.get('event_id', 'unknown')
        etype = evt.get('event_type', 'unknown')
        severity = evt.get('severity', 0)
        channels = ', '.join(evt.get('affected_channels', []))
        ts = evt.get('timestamp_formatted', '')
        print(f'  [{i}] {eid}: {etype} (severity={severity:.2f}) on {channels}')
        if ts:
            print(f'       {ts}')
except Exception as e:
    print(f'Could not fetch events: {e}')
    events = []

## 3. Analyze an Event with AI

Select an event and submit it to the RAG API for analysis. The AI uses
domain knowledge about power quality standards, equipment behavior, and
root cause patterns to produce a structured analysis.

In [ ]:
# Select an event to analyze (change index to pick a different event)
EVENT_INDEX = 0

if events:
    selected_event = events[EVENT_INDEX]
    print(f"Selected event: {selected_event.get('event_id')}")
    print(f"Type: {selected_event.get('event_type')}")
    print(f"Severity: {selected_event.get('severity')}")
    print(f"Channels: {selected_event.get('affected_channels')}")
    print()
    print('Event data:')
    print(json.dumps(selected_event, indent=2, default=str))
else:
    print('No events available. Using a sample query instead.')
    selected_event = None

In [ ]:
def query_rag(question, device_id=None, session_id='notebook'):
    """Send a query to the RAG API and return the response."""
    payload = {
        'query': question,
        'session_id': session_id,
    }
    if device_id:
        payload['device_id'] = device_id

    r = requests.post(
        f'{RAG_BASE_URL}/query',
        json=payload,
        timeout=120,
    )
    r.raise_for_status()
    return r.json()


# Build analysis query from selected event
if selected_event:
    event_summary = (
        f"Analyze this power quality event:\n"
        f"Type: {selected_event.get('event_type')}\n"
        f"Severity: {selected_event.get('severity')}\n"
        f"Affected channels: {selected_event.get('affected_channels')}\n"
        f"Timestamp: {selected_event.get('timestamp_formatted', 'unknown')}\n"
    )
    metadata = selected_event.get('metadata', {})
    if metadata:
        event_summary += f"Metadata: {json.dumps(metadata)}\n"

    event_summary += (
        "\nProvide: severity assessment, likely root cause, "
        "potential equipment impact, and recommended actions."
    )
else:
    event_summary = (
        "A voltage sag of 15% on phases A and B was detected, lasting 200ms. "
        "The facility has a 480V distribution system with VFDs and sensitive "
        "semiconductor manufacturing equipment. "
        "Provide: severity assessment, likely root cause, "
        "potential equipment impact, and recommended actions."
    )

print('Query:')
print(textwrap.fill(event_summary, width=80))

In [ ]:
# Submit to RAG API
print('Submitting to AI for analysis...\n')

try:
    device_id = None
    if selected_event:
        device_id = selected_event.get('metadata', {}).get('device_id')

    response = query_rag(event_summary, device_id=device_id)

    # Display response metadata
    meta = response.get('metadata', {})
    if meta:
        print(f"Route: {meta.get('route', 'unknown')}")
        print(f"Model: {meta.get('model', 'unknown')}")
        print(f"Tokens: {meta.get('total_tokens', 'unknown')}")
        print(f"Time: {meta.get('execution_time_ms', 'unknown')} ms")
        print()

    # Display the analysis
    answer = response.get('answer', response.get('response', ''))
    print('--- AI Analysis ---')
    print(answer)

except requests.exceptions.ConnectionError:
    print('RAG API not available.')
    print('If running locally, request demo access at https://demo.pq.app')
except Exception as e:
    print(f'Analysis failed: {e}')

## 4. Generate Incident Report

The RAG API can generate a structured incident report suitable for
documentation, compliance, or customer communication.

In [ ]:
def generate_report(device_id=None, session_id='notebook'):
    """Request an incident report from the RAG API."""
    payload = {
        'session_id': session_id,
    }
    if device_id:
        payload['device_id'] = device_id

    r = requests.post(
        f'{RAG_BASE_URL}/report',
        json=payload,
        timeout=120,
    )
    r.raise_for_status()
    return r.json()


try:
    device_id = None
    if selected_event:
        device_id = selected_event.get('metadata', {}).get('device_id')

    report = generate_report(device_id=device_id)

    report_text = report.get('report', report.get('response', ''))
    print('--- Incident Report ---')
    print(report_text)

except requests.exceptions.ConnectionError:
    print('RAG API not available.')
except Exception as e:
    print(f'Report generation failed: {e}')

## 5. Interactive Analysis

Ask follow-up questions in the same session. The RAG system maintains
conversation context, so you can drill into specifics without repeating
the event details.

In [ ]:
# Follow-up questions (edit and re-run this cell)
followup = "What IEEE standards apply to this type of event, and are we within limits?"

try:
    response = query_rag(followup)
    print(response.get('answer', response.get('response', '')))
except Exception as e:
    print(f'Query failed: {e}')

In [ ]:
# Another follow-up
followup2 = "What monitoring changes would help catch this earlier next time?"

try:
    response = query_rag(followup2)
    print(response.get('answer', response.get('response', '')))
except Exception as e:
    print(f'Query failed: {e}')

## 6. Batch Analysis

Analyze multiple events to identify patterns or recurring issues.

In [ ]:
if len(events) >= 3:
    # Summarize recent events for pattern analysis
    event_summaries = []
    for evt in events[:10]:
        event_summaries.append(
            f"- {evt.get('event_type')} (severity={evt.get('severity', 0):.2f}) "
            f"on {', '.join(evt.get('affected_channels', []))} "
            f"at {evt.get('timestamp_formatted', 'unknown')}"
        )

    pattern_query = (
        f"Here are the {len(event_summaries)} most recent power quality events:\n\n"
        + '\n'.join(event_summaries)
        + "\n\nIdentify any patterns, recurring issues, or correlations. "
        "Are these events related? What is the most likely systemic cause?"
    )

    print('Pattern analysis query:')
    print(textwrap.fill(pattern_query, width=80))
    print()

    try:
        response = query_rag(pattern_query, session_id='batch-analysis')
        print('--- Pattern Analysis ---')
        print(response.get('answer', response.get('response', '')))
    except Exception as e:
        print(f'Pattern analysis failed: {e}')
else:
    print('Not enough events for pattern analysis (need at least 3).')

## Next Steps

- **Combine with waveform data**: Load the CPOW data around an event timestamp
  using `equser.data.load_cpow_scaled()` and include waveform characteristics
  in your AI query for deeper analysis.
- **Harmonic context**: Run the harmonic analysis notebook on event-adjacent
  data, then ask the AI about harmonic trends.
- **Scheduled analysis**: Use `equser.api.SynapseClient` in a script to
  periodically query events and generate automated reports.
- **Custom queries**: The RAG system understands power quality domain
  terminology. Ask about IEEE 519 compliance, CBEMA/ITIC curves,
  capacitor switching transients, VFD harmonics, or any other PQ topic.